In [1]:
import pyzx as zx
import pyzx_param as param
import random
import verify_t_gate as gbg

In [8]:
q, d, seed = 20, 50, 42
graph = zx.generate.cliffordT(q,d,seed=seed)
zx.draw(graph,scale=20)

In [2]:
circ = gbg.build_noisy_circuit(0.001)
g = circ.get_graph()

In [3]:
# tsim's built-in detector sampler replaces the hand-rolled weak_sim below.
# Default strategy "cat5" is the same stabilizer-rank decomposition
# gate_by_gate uses, so this timing is a like-for-like benchmark.
import time

t0 = time.perf_counter()
detector_sampler = circ.compile_detector_sampler()
print(f"compile: {time.perf_counter() - t0:.2f}s  (one-time)")

compile: 10.35s  (one-time)


In [12]:
# The first .sample() call JIT-compiles the XLA kernels; warm it up so the
# measured time reflects steady-state per-sample cost, not compilation.
detector_sampler.sample(shots=120000)

shots = 120000
t0 = time.perf_counter()
samples = detector_sampler.sample(shots=shots)
elapsed = time.perf_counter() - t0
print(f"{shots} shots in {elapsed:.4f}s  ->  {elapsed / shots * 1e3:.4f} ms/sample")
samples.shape

120000 shots in 7.6203s  ->  0.0635 ms/sample


(120000, 32)